# Predict Seagrass at different times
* Designed to work year by year
* Pick a location and run

In [1]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:54573/status,
Dashboard: http://127.0.0.1:54573/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:54574,Workers: 0
Dashboard: http://127.0.0.1:54573/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:53340,Total threads: 2
Dashboard: http://127.0.0.1:53341/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:54577,


# Value to edit

In [3]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
large_seagrass_sites =  ["Ihutai", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "LeftBank_Nov25", "ThePoint_Nov25"]
large_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "LeftBank_Nov25", "LeftBank_Feb26", "ThePoint_Nov25", "ThePoint_Feb26"]


In [4]:
predict_max_cloud_covers = [1, 0] # percentage

In [5]:
prediction_sites = all_training_sites
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [6]:
data_path = utils.get_data_path()
utils.create_data_folders()

mndwi_file = data_path / "mndwi_low_tide_thresholds.csv"

# Cells to run
* Get lowtide satellite images by year and site
* Predict Seagrass across satellite images

### Save satellite
The low-tide no cloud images across sites for a whole year

In [13]:
satellite_file

WindowsPath('C:/Local/repos/seagrass-detection/data/predictions/cloud_1_filtered/satellite_images/CatlinsRiverMouth_2018_sentinel-2.nc')

In [ ]:
all_baselines = sentinel2.load_band_quality_baseline()
mndwi_thresholds = pandas.read_csv(mndwi_file)

for max_cloud_cover in predict_max_cloud_covers:
    print(f"Max cloud percentage {max_cloud_cover}, MNDWI filtered low tide")
    _, satellite_path = utils.get_prediction_path(
        sample_method="sampling_2", method_2_threshold=0.5,
        model_max_cloud_cover=max_cloud_cover,
        predict_max_cloud_cover=max_cloud_cover,
        use_threshold=True, satellite_only=True,
        filtered=True)
    
    for prediction_site in prediction_sites:

        site_baseline = all_baselines.loc[all_baselines["site"] == prediction_site]
        mndwi_threshold = mndwi_thresholds.loc[mndwi_thresholds["site"] == prediction_site, "threshold",].squeeze()
        if not isinstance(mndwi_threshold, float):
            print(f"WARNING: Expected a float tide threshold for {site}. Exiting.")
            break
        if len(site_baseline) != len(sentinel2.RGB_BANDS):
            print(f"WARNING: Expected a float tide threshold for {site}. Exiting.")
            break

        site_polygon_file = utils.get_site_polygon_path(prediction_site)
        site_polygon = geopandas.read_file(site_polygon_file)
        
        for year in years:
            print(f"{prediction_site}: year {year}")
        
            satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"

            if (satellite_path / f"{prediction_site}_{years[-1]}_sentinel-2.nc").exists():
                print(f"\tAlready run for {prediction_site} as {prediction_site}_{years[-1]}_sentinel-2.nc exists. "
                      "Delete if you want redownloaded.")
                break
        
            if not satellite_file.exists():
                print(f"\tSave satellite image of the site {prediction_site} around lowtide without cloud in the year {year}")
                for i in range(2):
                    try:
                        satellite_data = sentinel2.get_mndwi_and_cloud_filtered_images_in_year(
                            geometry=site_polygon,
                            max_cloud_cover=max_cloud_cover,
                            year=year,
                            mndwi_threshold=mndwi_threshold,
                            anomaly_filter_baseline=site_baseline,
                            n_mad=3,
                        )
                        break
                    except Exception:
                        print("Caught exception downloading images and try again.")
                if len(satellite_data['time']) == 0:
                    print("Warning: No satellite data without cloud. Ignore")
                else:
                    print("\t\tSave")
                    utils.write_netcdf_conventions_in_place(satellite_data)
                    satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                    satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                    utils.write_netcdf_conventions_in_place(satellite_data)
                    utils.save_netcdf(satellite_data, satellite_file)
                utils.rgb_from_satellite(satellite_data_path=satellite_file)

Max cloud percentage 1, MNDWI filtered low tide
Ihutai: year 2016
	Save satellite image of the site Ihutai around lowtide without cloud in the year 2016
	mean MNDWI less than 0.2859899044036865 filtered tiles: 12 from 53 STAC items
	Less than 1% cloud tiles: 2 from the MNDWI filtered tiles of 12. Cloud percentages [96.4 99.   0.7 60.5 57.3  1.1 99.1 88.3 86.2 99.9  0.2 98.8] Kept [False False  True False False False False False False False  True False]
		2016-02-19: B04=0.49, B03=0.29, B02=0.08 -> kept
		2016-11-25: B04=2.02, B03=1.49, B02=1.08 -> kept
	Anomaly filter: 2 of 2 dates kept for site Ihutai (n_mad=3)
		Save
Ihutai: year 2017
	Save satellite image of the site Ihutai around lowtide without cloud in the year 2017
	mean MNDWI less than 0.2859899044036865 filtered tiles: 18 from 44 STAC items
	Less than 1% cloud tiles: 3 from the MNDWI filtered tiles of 18. Cloud percentages [ 0.3  0.5 99.1 85.7 20.4 55.1 89.8 93.3 85.8 66.2 97.  66.8 95.7 89.6
 78.3  0.3  3.9  1.3] Kept [ True 